In [ ]:
!pip -q install kaggle catboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.0 MB/s eta 0:00:00


In [ ]:
import os, json

KAGGLE_USERNAME = input("Enter your Kaggle username: ").strip()
KAGGLE_KEY = input("Enter your Kaggle API key: ").strip()

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle API configured successfully.")

Enter your Kaggle username: aadityatiwary2004
Enter your Kaggle API key: KGAT_6f4fe5f62c6a1af193231e1906b188f5
Kaggle API configured successfully.


In [ ]:
!kaggle datasets download -d ifteshanajnin/carinsuranceclaimprediction-classification -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/ifteshanajnin/carinsuranceclaimprediction-classification
License(s): other
100% 1.96M/1.96M [00:00<00:00, 158MB/s]



In [ ]:
import os
import pandas as pd

data_path = "/content/data"

csv_files = []
for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print("CSV files found:")
for f in csv_files:
    print(f)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV file found in the downloaded dataset.")

# Load the 'train.csv' file instead of csv_files[0]
train_csv_path = None
for f in csv_files:
    if "train.csv" in f:
        train_csv_path = f
        break

if train_csv_path is None:
    raise FileNotFoundError("train.csv not found in the downloaded dataset.")

df = pd.read_csv(train_csv_path)
print("\nLoaded file:", train_csv_path)
print("Shape:", df.shape)
df.head()

CSV files found:
/content/data/sample_submission.csv
/content/data/test.csv
/content/data/train.csv

Loaded file: /content/data/train.csv
Shape: (58592, 44)


,policy_id,policy_tenure,age_of_car,age_of_policyholder,area_cluster,population_density,make,segment,model,fuel_type,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,is_claim
0,ID00001,0.515874,0.05,0.644231,C1,4990,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
1,ID00002,0.672619,0.02,0.375000,C2,27003,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
2,ID00003,0.841110,0.02,0.384615,C3,4076,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
3,ID00004,0.900277,0.11,0.432692,C4,21622,1,C1,M2,Petrol,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0
4,ID00005,0.596403,0.11,0.634615,C5,34738,2,A,M3,Petrol,...,No,Yes,Yes,Yes,No,Yes,Yes,Yes,2,0


In [ ]:
print("Columns:\n")
for col in df.columns:
    print(col)

print("\nData types:\n")
print(df.dtypes)

print("\nMissing values:\n")
print(df.isnull().sum())

Columns:

policy_id
policy_tenure
age_of_car
age_of_policyholder
area_cluster
population_density
make
segment
model
fuel_type
max_torque
max_power
engine_type
airbags
is_esc
is_adjustable_steering
is_tpms
is_parking_sensors
is_parking_camera
rear_brakes_type
displacement
cylinder
transmission_type
gear_box
steering_type
turning_radius
length
width
height
gross_weight
is_front_fog_lights
is_rear_window_wiper
is_rear_window_washer
is_rear_window_defogger
is_brake_assist
is_power_door_locks
is_central_locking
is_power_steering
is_driver_seat_height_adjustable
is_day_night_rear_view_mirror
is_ecw
is_speed_alert
ncap_rating
is_claim

Data types:

policy_id                            object
policy_tenure                       float64
age_of_car                          float64
age_of_policyholder                 float64
area_cluster                         object
population_density                    int64
make                                  int64
segment                              objec

In [ ]:
import numpy as np

possible_targets = [
    "is_claim", "claim", "target", "Claim", "Is_Claim", "Response",
    "made_claim", "claim_status", "has_claim"
]

target_col = None
for col in df.columns:
    if col in possible_targets:
        target_col = col
        break

if target_col is None:
    for col in df.columns:
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) == 2:
            target_col = col
            break

if target_col is None:
    raise ValueError("Could not auto-detect target column. Please inspect the dataset manually.")

print("Detected target column:", target_col)

df = df.drop_duplicates().copy()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "Unknown")
    else:
        df[col] = df[col].fillna(df[col].median())

X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

Detected target column: is_claim
X shape: (58592, 43)
y distribution:
 is_claim
0    54844
1     3748
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from catboost import CatBoostClassifier

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        verbose=0,
        random_state=42,
        iterations=200,
        depth=6,
        learning_rate=0.05
    ))
])

model.fit(X_train, y_train)
print("Model training completed.")

Model training completed.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}\n")

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9360
ROC-AUC : 0.6514

Classification Report:

              precision    recall  f1-score   support

           0       0.94      1.00      0.97     10969
           1       0.00      0.00      0.00       750

    accuracy                           0.94     11719
   macro avg       0.47      0.50      0.48     11719
weighted avg       0.88      0.94      0.91     11719

Confusion Matrix:

[[10969     0]
 [  750     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
results = X_test.copy()
results["Actual_Claim"] = y_test.values
results["Claim_Probability"] = y_prob
results["Trust_Score"] = ((1 - results["Claim_Probability"]) * 100).round(2)

def trust_level(score):
    if score >= 80:
        return "High Trust"
    elif score >= 60:
        return "Medium Trust"
    else:
        return "Low Trust"

def premium_band(score):
    if score >= 80:
        return "Low Premium"
    elif score >= 60:
        return "Standard Premium"
    else:
        return "High Premium"

results["Trust_Level"] = results["Trust_Score"].apply(trust_level)
results["Premium_Band"] = results["Trust_Score"].apply(premium_band)

results.head(10)

,policy_id,policy_tenure,age_of_car,age_of_policyholder,area_cluster,population_density,make,segment,model,fuel_type,...,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,Actual_Claim,Claim_Probability,Trust_Score,Trust_Level,Premium_Band
38215,ID38216,0.973237,0.03,0.423077,C5,34738,3,C2,M4,Diesel,...,Yes,No,Yes,Yes,3,0,0.086938,91.31,High Trust,Low Premium
3597,ID03598,0.020455,0.04,0.538462,C10,73430,1,A,M1,CNG,...,No,No,No,Yes,0,0,0.028271,97.17,High Trust,Low Premium
32004,ID32005,0.429325,0.01,0.394231,C7,6112,1,A,M1,CNG,...,No,No,No,Yes,0,0,0.044754,95.52,High Trust,Low Premium
20540,ID20541,1.059081,0.02,0.307692,C13,5410,3,C2,M4,Diesel,...,Yes,No,Yes,Yes,3,0,0.087144,91.29,High Trust,Low Premium
55338,ID55339,0.083045,0.09,0.317308,C5,34738,1,C1,M2,Petrol,...,Yes,Yes,Yes,Yes,2,0,0.035863,96.41,High Trust,Low Premium
42587,ID42588,0.759958,0.03,0.375000,C9,17804,1,B2,M6,Petrol,...,Yes,Yes,Yes,Yes,2,0,0.081954,91.80,High Trust,Low Premium
9788,ID09789,0.075651,0.03,0.519231,C10,73430,1,B2,M6,Petrol,...,Yes,Yes,Yes,Yes,2,0,0.033656,96.63,High Trust,Low Premium
14168,ID14169,0.083657,0.04,0.375000,C15,290,3,C2,M4,Diesel,...,Yes,No,Yes,Yes,3,0,0.032569,96.74,High Trust,Low Premium
21156,ID21157,0.006241,0.07,0.471154,C8,8794,3,C2,M4,Diesel,...,Yes,No,Yes,Yes,3,0,0.033383,96.66,High Trust,Low Premium
36919,ID36920,0.948750,0.00,0.423077,C7,6112,1,B2,M6,Petrol,...,Yes,Yes,Yes,Yes,2,0,0.067814,93.22,High Trust,Low Premium


In [ ]:
print(results[["Claim_Probability", "Trust_Score", "Trust_Level", "Premium_Band"]].head(15))

       Claim_Probability  Trust_Score Trust_Level Premium_Band
38215           0.086938        91.31  High Trust  Low Premium
3597            0.028271        97.17  High Trust  Low Premium
32004           0.044754        95.52  High Trust  Low Premium
20540           0.087144        91.29  High Trust  Low Premium
55338           0.035863        96.41  High Trust  Low Premium
42587           0.081954        91.80  High Trust  Low Premium
9788            0.033656        96.63  High Trust  Low Premium
14168           0.032569        96.74  High Trust  Low Premium
21156           0.033383        96.66  High Trust  Low Premium
36919           0.067814        93.22  High Trust  Low Premium
44101           0.121204        87.88  High Trust  Low Premium
32153           0.026104        97.39  High Trust  Low Premium
35854           0.033028        96.70  High Trust  Low Premium
31264           0.031872        96.81  High Trust  Low Premium
13033           0.051469        94.85  High Trust  Low 

In [ ]:
sample_driver = {}

for col in X.columns:
    if col in num_cols:
        sample_driver[col] = float(input(f"Enter value for {col}: "))
    else:
        sample_driver[col] = input(f"Enter value for {col}: ")

sample_df = pd.DataFrame([sample_driver])

sample_prob = model.predict_proba(sample_df)[:, 1][0]
sample_trust = round((1 - sample_prob) * 100, 2)

if sample_trust >= 80:
    sample_level = "High Trust"
    sample_premium = "Low Premium"
elif sample_trust >= 60:
    sample_level = "Medium Trust"
    sample_premium = "Standard Premium"
else:
    sample_level = "Low Trust"
    sample_premium = "High Premium"

print("\n===== INSUREDGE TRUST SCORE RESULT =====")
print(f"Claim Probability : {sample_prob:.4f}")
print(f"Trust Score       : {sample_trust}/100")
print(f"Trust Level       : {sample_level}")
print(f"Premium Band      : {sample_premium}")

Enter value for policy_id: 1
Enter value for policy_tenure: 2
Enter value for age_of_car: 4
Enter value for age_of_policyholder: 30
Enter value for area_cluster: 31
Enter value for population_density: 4
Enter value for make: 7
Enter value for segment: 9
Enter value for model: 89
Enter value for fuel_type: 2
Enter value for max_torque: 90
Enter value for max_power: 90
Enter value for engine_type: 5
Enter value for airbags: 4
Enter value for is_esc: 9
Enter value for is_adjustable_steering: 8
Enter value for is_tpms: 9
Enter value for is_parking_sensors: 9
Enter value for is_parking_camera: 9
Enter value for rear_brakes_type: 9
Enter value for displacement: 9
Enter value for cylinder: 9
Enter value for transmission_type: 9
Enter value for gear_box: 9
Enter value for steering_type: 9
Enter value for turning_radius: 9
Enter value for length: 9
Enter value for width: 9
Enter value for height: 9
Enter value for gross_weight: 9
Enter value for is_front_fog_lights: 9
Enter value for is_rear_wi

In [ ]:
import joblib

joblib.dump(model, "/content/insuredge_trust_score_model.pkl")
results.to_csv("/content/insuredge_trust_score_results.csv", index=False)

print("Saved model: /content/insuredge_trust_score_model.pkl")
print("Saved results: /content/insuredge_trust_score_results.csv")

Saved model: /content/insuredge_trust_score_model.pkl
Saved results: /content/insuredge_trust_score_results.csv


In [ ]:
import pickle
from google.colab import files
with open('insuredge_trust_score_model.pkl', 'wb') as f:
    pickle.dump(model, f)
files.download('insuredge_trust_score_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>